# ☀️ Pronóstico operativo solar — cuaderno de resultados

Planta de referencia **Medellín, 500 kWp DC / 400 kW AC**, pronóstico a día
adelante emitido a las 12:00 UTC del día anterior.

> **Lea esto antes que los números.** Las cifras de este cuaderno salen de una
> planta **sintética**, no de una planta real. Sirven para mostrar el método y
> el formato del resultado, no para estimar el desempeño en un sitio concreto.
> Además, la métrica y el horizonte dependen de decisiones que todavía no se
> han tomado (Fase 1). Vea la última sección.

Este cuaderno **no entrena nada**: lee artefactos ya calculados. Corre en
segundos.

In [ ]:
# Configuración. Cambie REPO por la URL de su repositorio (ver CONFIG.md).
REPO = "https://github.com/orozcoig/pronostico-solar"

import os, sys, subprocess
from pathlib import Path

NOMBRE = REPO.rstrip("/").split("/")[-1]
if "<usuario>" in REPO and not Path("artefactos").exists():
    raise SystemExit(
        "Falta configurar REPO con la URL real del repositorio. Ver CONFIG.md."
    )
if not Path(NOMBRE).exists() and not Path("artefactos").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)

RAIZ = Path(NOMBRE) if Path(NOMBRE).exists() else Path(".")
sys.path.insert(0, str(RAIZ))
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas", "numpy", "pyarrow", "matplotlib", "pvlib", "scikit-learn"],
               check=False)

import json
import pandas as pd

ART = RAIZ / "artefactos"
if not (ART / "resumen.json").exists():
    raise SystemExit(
        f"No encuentro artefactos en {ART}. El repositorio debe traerlos versionados; "
        "si no, regenérelos con banco.artefactos.generar(). Ver CONFIG.md."
    )
pred = pd.read_parquet(ART / "predicciones.parquet")
conj = pd.read_parquet(ART / "conjunto.parquet")
res  = json.loads((ART / "resumen.json").read_text("utf-8"))
print(f"{res['dias_evaluados']} días evaluados · {len(pred):,} horas · listo en segundos")

## 1. Lo que pasó: generación pronosticada contra la real

La línea café es lo que la planta produjo. La azul es el pronóstico. La banda
sombreada es el rango en el que esperamos que caiga la generación 8 de cada 10
veces.

**La banda es ancha a propósito.** No es falta de ajuste: es la incertidumbre
real de pronosticar un sitio único con un día de anticipación. Una nube mueve
el cien por ciento de una planta, no una fracción del país.

In [ ]:
from banco.artefactos import figura_pronostico
from banco.sitios import MEDELLIN_REF as sitio
figura_pronostico(pred, sitio, dias=10)

## 2. Lectura en lenguaje de negocio

Cuatro números y qué significan.

In [ ]:
n = res
print(f"""
ERROR TÍPICO
  El pronóstico se equivoca, en promedio, en {n['mae_modelo_kw']:.0f} kW por hora de sol.
  Eso es el {n['nmae_modelo']:.0%} de la generación media diurna ({n['generacion_media_diurna_kw']:.0f} kW).

CONTRA QUÉ SE COMPARA
  La mejor alternativa simple (un modelo físico, sin aprendizaje) se equivoca
  en {n['mae_mejor_linea_base_kw']:.0f} kW. La mejora es de {n['mejora_sobre_linea_base_kw']:.1f} kW.
  El margen de error de esa comparación ("la vara") es {n['vara_kw']:.1f} kW,
  así que la mejora {'SÍ' if n['supera_la_vara'] else 'NO'} es distinguible del ruido.

CUÁNTO DEL ERROR SE PUEDE COMPRAR
  Si el pronóstico del tiempo fuera perfecto, el error bajaría a {n['mae_techo_alcanzable_kw']:.0f} kW.
  Es decir: el {n['fraccion_error_importada_del_meteorologico']:.0%} del error NO viene del modelo,
  viene del proveedor meteorológico. Mejorar el modelo compite por el {1-n['fraccion_error_importada_del_meteorologico']:.0%} restante.

CUÁNTA INCERTIDUMBRE HAY QUE PLANEAR
  La banda de confianza mide {n['ancho_mediano_banda_kw']:.0f} kW de ancho ({n['ancho_banda_sobre_placa']:.0%} de la capacidad),
  y acierta el {n['cobertura_banda_10_90']:.0%} de las veces (objetivo 80 %).
""")

### Por qué el tercer número es el más importante

Si el 87 % del error viene del proveedor meteorológico, **ninguna inversión en
modelado lo toca**. La conversación útil es sobre el insumo: post-procesar el
pronóstico, mezclar fuentes, usar un ensamble. Cualquier propuesta que prometa
reducir el error sin mencionar esto está prometiendo algo que no controla.

## 3. Diagnóstico: qué encontramos en los datos y en el modelo

Dos revisiones automáticas. La primera busca problemas en la **medición y la
operación** de la planta; la segunda, en el **modelo**. Ambas pueden —y deben—
concluir que no hay nada que corregir.

In [ ]:
from banco.diagnostico_modelo import diagnosticar_modelo, informe
col = "q50" if "q50" in pred.columns else "modelo_fisico"
ds = diagnosticar_modelo(pred["real"], pred[col], conj, sitio,
                         mae_con_observado=res["mae_techo_alcanzable_kw"])
print(informe(ds))

Cada hallazgo trae **qué aplicar, qué NO aplicar, y qué debería pasar si el
diagnóstico es correcto**. Esa última parte es lo que lo hace verificable: si
se aplica la técnica y no ocurre lo previsto, el diagnóstico estaba mal.

In [ ]:
# Las dos vistas que delatan problemas de geometría u obstrucción.
from banco.artefactos import figura_residuo
figura_residuo(pred, conj, sitio)

## 4. Cargar los datos de su planta

El cuaderno lee a través de un **contrato de datos** único. Cambiar de la
planta sintética a su planta es cambiar la carpeta, no el código.

Formato esperado (hora local, sin zona horaria, como lo exporta un historiador):

- `sitio.json` — capacidad, inclinación, azimut, ubicación, punto de medición
- `generacion.csv` — `instante, potencia_ac_kw`
- `clima_observado.csv` — `instante, ghi_wm2, dni_wm2, dhi_wm2, temp_aire_c, viento_ms`
- `clima_pronosticado.csv` — igual, más `emitido_en` y `horizonte_h`

In [ ]:
# from banco.fuente_csv import FuenteCSV
# from banco.agente import diagnosticar
# f = FuenteCSV("mis_datos/")
# meta = f.metadatos_sitio()
# u, v = pd.Timestamp("2023-01-01", tz="UTC"), pd.Timestamp("2025-01-01", tz="UTC")
# ahora = pd.Timestamp.now(tz="UTC")
# hallazgos = diagnosticar(f.generacion(u, v, ahora), f.clima_observado(u, v), meta)
# print([h.falla for h in hallazgos] or "sin hallazgos: no hay nada que corregir")

### Qué hace falta para que esto sea útil con sus datos

Hay un criterio de entrada, y se evalúa con números antes de cotizar nada.

In [ ]:
from banco.criterios import m1_historia_minima, m2_archivo_de_pronostico, m7_historia_vs_falla, tabla
# Ajuste estos valores a su caso:
MESES_MEDICION, RESOLUCION_MIN, MESES_ARCHIVO = 24, 5, 24
FALLA_QUE_LE_PREOCUPA = "deriva_suciedad"
print(tabla([m1_historia_minima(MESES_MEDICION, RESOLUCION_MIN),
             m2_archivo_de_pronostico(MESES_ARCHIVO),
             m7_historia_vs_falla(MESES_MEDICION, FALLA_QUE_LE_PREOCUPA)]))

## 5. Qué no sabemos todavía, y qué cambiaría

Estos resultados descansan sobre decisiones que **usted** tiene que tomar, no
nosotros. Mientras no estén tomadas, ningún número de arriba es transferible a
una propuesta.

| Pregunta | Qué supusimos | Qué cambia si la respuesta es otra |
|---|---|---|
| ¿A qué hora hay que emitir el pronóstico y para qué trámite? | 12:00 UTC del día anterior, día adelante | Si es intradiario (0–6 h) manda la persistencia, no el modelo del tiempo, y el 87 % de error importado se desploma |
| ¿Cuánto cuesta pasarse y cuánto quedarse corto? | Que cuestan lo mismo | Si son distintos, el pronóstico óptimo no es la mediana sino un cuantil. Elegir mal el cuantil cuesta más que toda la mejora de modelo lograda |
| ¿Qué cantidad se liquida? | La generación bruta a la salida del inversor | Si es la exportación (generación menos autoconsumo), el objetivo del modelo cambia entero |
| ¿Qué sistema consume el pronóstico? | Ninguno definido | Un pronóstico que ningún sistema ingiere vale cero |
| ¿Hay archivo de corridas pasadas del proveedor? | 24 meses | Sin archivo cambia el cronograma. Entrenar con reanálisis en su lugar es la fuga más cara de este proyecto |

El detalle completo está en `DEPENDENCIAS_FASE1.md`.